# DNA Core — Extraction + Matching Pipeline  v4  (Fiserv Foundation API)

Extracts dollar line items from DNA contract PDFs and matches them to the DNA material code dictionary.  
**v4 changes:** OpenAI SDK removed; all AI calls routed through the Fiserv Foundation keyless proxy via `httpx`.

## ⚠ Pre-flight checklist — read before running

| # | Check | Detail |
|---|-------|--------|
| 1 | **Fiserv network / VPN** | Proxy URL is internal; script will time-out off-network |
| 2 | **`EMAIL_ID`** in Cell 3 | Set to your actual `@fiserv.com` address |
| 3 | **`marked_checkbox_example.png`** | Must exist at `BASE_DIR`; Phase 1 crashes without it |
| 4 | **DPI** | Reduced 600 → 150 to keep base64 payloads within proxy limits.  Raise to 200–300 if text is unreadable, but expect slower runs |
| 5 | **CHUNK_WORKERS** | Default 4; reduce to 1–2 if proxy timeouts occur on large PDFs |
| 6 | **MAX_PARALLEL_CALLS** | Default 22; reduce to 10–15 if Phase 2 hits proxy rate limits |
| 7 | **Run order** | Cell 1 (install once) → Cells 2–9 (load all functions) → Cell 10 (discovery) → Cell 11 (Phase 1) → Cell 12 (Phase 2) |


In [ ]:
# Run once; restart kernel after install if packages were missing
# !pip install pandas rapidfuzz openpyxl httpx sentence-transformers scikit-learn numpy PyMuPDF Pillow

In [ ]:
import sys
import os
sys.stdout.reconfigure(encoding='utf-8')
sys.stderr.reconfigure(encoding='utf-8')
import json
import base64
import math
import time
import re
import threading
import uuid
import httpx
import numpy as np
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from collections import defaultdict

import fitz
fitz.TOOLS.mupdf_display_errors(False)
import pandas as pd
from PIL import Image
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from rapidfuzz import fuzz, process

print('Imports OK')

In [ ]:
DNA_DIR         = r"C:\Users\F3MWEW7\OneDrive - Fiserv Corp\Documents\Scripts"
CONTRACTS_DIR   = r"C:\Users\F3MWEW7\OneDrive - Fiserv Corp\Documents\Contracts"
DICTIONARY_FILE = r"C:\Users\F3MWEW7\OneDrive - Fiserv Corp\Documents\Scripts\Inputs\DNA Dictionary US 03-26.xlsx"
NORM_FILE       = r"C:\Users\F3MWEW7\OneDrive - Fiserv Corp\Documents\Scripts\Inputs\DNA Section Headers Normalization.xlsx"

OUTPUT_DIR         = r"C:\Users\F3MWEW7\OneDrive - Fiserv Corp\Documents\Scripts\Output\Extraction"
MATCHED_OUTPUT_DIR = r"C:\Users\F3MWEW7\OneDrive - Fiserv Corp\Documents\Scripts\Output\Matched"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MATCHED_OUTPUT_DIR, exist_ok=True)

# ── Proxy config (no API key needed) ────────────────────────────────────
EMAIL_ID = 'arinis.naskar@fiserv.com'   # ← UPDATE to your actual @fiserv.com address

# ── Run control ─────────────────────────────────────────────────────────
ONLY_CLIENT = None   # set to 'CLIENT NAME' to process one client; None = all

# ── Model / extraction settings ─────────────────────────────────────────
# EXTRACTION_MODEL removed — proxy routes to its default deployment
MATCHING_MODEL   = 'gpt-5.1'  # informational; proxy routes to its default deployment
CHUNK_SIZE       = 8
DPI              = 600   # full resolution; relies on _proxy_post retry/backoff to ride out 504s

DATE_CUTOFF_YEAR = 2022
CHUNK_WORKERS    = 2     # parallel chunks per PDF; raise to 4 only if proxy is stable

ITEM_BATCH_SIZE             = 25
DICT_CHUNK_SIZE             = 250
MAX_PARALLEL_CALLS          = 12   # Phase 2 threads; raise to 22 only if proxy is stable
FUZZY_AUTO_ACCEPT_THRESHOLD = 0.90
SEMANTIC_WEIGHT             = 0.6
LEXICAL_WEIGHT              = 0.4
SECTION_NORM_THRESHOLD      = 92

print('Config loaded.')

In [ ]:
FOUNDATION_API_URL = (
    'https://dev-cst-cognitive-service.onefiserv.net'
    '/FoundationAPI/openai/deployments/default/chat/completions'
    '?api-version=2025-03-01-preview'
)

_SESSION_ID = str(uuid.uuid4())   # shared across all calls in this kernel session


def _make_headers():
    return {
        'Content-Type': 'application/json',
        'X-Purpose':    'GPT5.1Purpose',
        'X-Session-Id': _SESSION_ID,
        'X-Source':     'PythonClient',
        'X-Email-Id':   EMAIL_ID,
    }


def _proxy_post(payload, retries=5, wait_secs=30):
    """POST to the Fiserv Foundation API; return the parsed response dict.
    Retries transient gateway errors (502/503/504), rate limits (429) and
    timeouts with exponential backoff (30, 60, 120, 240s) — tuned for the
    long processing time of high-DPI image payloads."""
    for attempt in range(1, retries + 1):
        try:
            resp = httpx.post(
                FOUNDATION_API_URL,
                json=payload,
                headers=_make_headers(),
                timeout=300,
            )
            resp.raise_for_status()
            return resp.json()
        except httpx.HTTPStatusError as e:
            status = e.response.status_code
            if status == 429 or status >= 500:
                wait = wait_secs * (2 ** (attempt - 1))
                print(f'  Proxy HTTP {status} (attempt {attempt}/{retries}), retrying in {wait}s...')
                if attempt < retries:
                    time.sleep(wait)
            else:
                print(f'  Proxy HTTP {status} (non-retryable): {e.response.text[:300]}')
                raise
        except (httpx.TimeoutException, httpx.NetworkError) as e:
            wait = wait_secs * (2 ** (attempt - 1))
            print(f'  Proxy network/timeout error (attempt {attempt}/{retries}): {e}, retrying in {wait}s...')
            if attempt < retries:
                time.sleep(wait)
    raise RuntimeError(f'Proxy call failed after {retries} retries')


print(f'Proxy session ID: {_SESSION_ID}')
print('Proxy helper ready.')

In [ ]:
def extract_date_from_filename(filename):
    pattern = r'(\d{1,2})-(\d{1,2})-(\d{4})'
    match = re.search(pattern, filename)
    if not match:
        return None
    month, day, year = match.groups()
    try:
        return datetime(int(year), int(month), int(day))
    except ValueError:
        return None


def pdf_to_images(pdf_path, dpi=DPI):
    doc = fitz.open(pdf_path)
    pages = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        mat  = fitz.Matrix(dpi / 72, dpi / 72)
        pix  = page.get_pixmap(matrix=mat)
        img  = Image.open(BytesIO(pix.tobytes('png')))
        pages.append({'page_number': i + 1, 'image': img})
    return pages


def image_to_base64(image):
    buf = BytesIO()
    image.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode('utf-8')


def clean_price(val):
    """Normalise a price string. Preserves percentage values, $0/Included/Waived labels."""
    if pd.isna(val):
        return ''
    s = str(val).strip()
    if not s:
        return ''
    s_low = s.lower()
    if any(lbl in s_low for lbl in ('included', 'waived', 'no charge', 'n/a', 'free', 'complimentary', 'tbd')):
        return s
    if '%' in s:
        return s
    if re.match(r'^over\s*\$', s, re.IGNORECASE):
        return s
    if re.match(r'^\$?\d+(\.\d+)?\s*[MBmb]$', s.strip()):
        return s
    negative = bool(re.match(r'^\(', s.replace('$', '').strip()))
    cleaned  = s.replace('$', '').replace(',', '').replace('(', '').replace(')', '').strip()
    m = re.search(r'\d+\.?\d*', cleaned)
    if not m:
        return s
    value = float(m.group())
    return f'-${value:,.2f}' if negative else f'${value:,.2f}'


def detect_pricing_type(val):
    """Classify a price value as flat / percentage / waived / included / tiered / tbd / unknown."""
    if pd.isna(val):
        return ''
    s = str(val).strip().lower()
    if not s:
        return ''
    if '%' in s:
        return 'percentage'
    if 'waived' in s:
        return 'waived'
    if 'included' in s:
        return 'included'
    if 'no charge' in s or 'free' in s or 'complimentary' in s or 'n/a' in s:
        return 'no charge'
    if 'tbd' in s or 'to be determined' in s:
        return 'tbd'
    if re.search(r'\d', s):
        return 'flat'
    return 'unknown'


TIER_CONTEXT_KEYWORDS = (
    'account', 'accounts', 'dda', 'ddas', 'share draft', 'share drafts',
    'transaction', 'transactions', 'user', 'users', 'seat', 'seats',
    'member', 'members', 'item', 'items', 'ach', 'card', 'cards',
    'deposit', 'deposits', 'loan', 'loans', 'statement', 'statements',
    'asset', 'assets', 'tier', 'tiers', 'subscriber', 'subscribers',
    'device', 'devices', 'branch', 'branches', 'core', 'cores',
    'customer', 'customers', 'client', 'clients', 'call', 'calls',
    'minute', 'minutes', 'month', 'months', 'page', 'pages',
    'record', 'records', 'report', 'reports', 'license', 'licenses',
)


def extract_tier_range(item_text):
    """Capture tier ranges like 'Up to 500 accounts' — only when tier context is present."""
    if pd.isna(item_text):
        return ''
    s = str(item_text).strip()
    if not s:
        return ''
    s_low = s.lower()
    if not any(kw in s_low for kw in TIER_CONTEXT_KEYWORDS):
        return ''
    m = re.search(
        r'(up to\s*[\d,]+(?:\s*(?:%s))?)' % '|'.join(TIER_CONTEXT_KEYWORDS),
        s, re.IGNORECASE,
    )
    if m:
        return m.group(1).strip()
    m = re.search(r'(\d{1,3}(?:,\d{3})+\s*(?:to|[-\u2013])\s*\d{1,3}(?:,\d{3})+)', s)
    if m:
        return m.group(1).strip()
    m = re.search(r'(\d{3,}\s*(?:to|[-\u2013])\s*\d{3,})', s)
    if m:
        return m.group(1).strip()
    m = re.search(r'(over\s*[\d,]+)', s, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return ''


def call_api_with_retry(fn, max_retries=5):
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            err = str(e).lower()
            if '429' in str(e) or 'rate limit' in err or 'too many requests' in err:
                wait = 20 * (2 ** attempt)
                print(f'  Rate limit hit (attempt {attempt+1}/{max_retries}), retrying in {wait}s...')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f'API call failed after {max_retries} retries')


FREQUENCY_KEYWORDS = {
    'One-Time': ['one time', 'one-time', 'onetime', 'set up', 'set-up', 'setup', 'implementation', 'implement'],
    'Monthly':  ['monthly', 'month'],
    'Annual':   ['annual', 'annually', 'yearly', 'year'],
}


def _apply_frequency_inference(df):
    """Fill blank frequency values by checking description for frequency keywords."""
    def _infer(row):
        freq = row.get('Frequency')
        if isinstance(freq, str) and freq.strip():
            return freq
        desc = str(row.get('Item') or '').lower()
        for label, keywords in FREQUENCY_KEYWORDS.items():
            if any(kw in desc for kw in keywords):
                return label
        return freq
    df = df.copy()
    df['Frequency'] = df.apply(_infer, axis=1)
    return df


print('Shared helpers loaded.')

In [ ]:
_MSA_NAME_KWS     = ('msa', 'master agreement', 'master-agreement', 'master service', 'master services', 'master-services-agreement')
_AMEND_NAME_KWS   = ('amendment', 'addendum', 'amend')
_RENEWAL_TEXT_KWS = ('renewal amendment', 'renewal and amendment', 'renewal & amendment')
_QUALIFYING_TYPES = {'MSA', 'Renewal_Amendment'}


def classify_contract(full_path):
    """
    Classify a PDF as MSA | Renewal_Amendment | Amendment | Other.
    MSA: filename keywords only.
    Renewal Amendment: filename indicates amendment AND ('Renewal Amendment' in first
                       3 pages of text OR document > 30 pages).
    """
    name_lower = os.path.basename(full_path).lower()
    if any(kw in name_lower for kw in _MSA_NAME_KWS):
        return 'MSA'
    if any(kw in name_lower for kw in _AMEND_NAME_KWS):
        first_text = ''
        page_count = 0
        try:
            doc = fitz.open(full_path)
            try:
                page_count = len(doc)
                first_text = ' '.join(
                    doc[i].get_text() for i in range(min(3, page_count))
                ).lower()
            finally:
                doc.close()
        except Exception:
            pass
        if any(kw in first_text for kw in _RENEWAL_TEXT_KWS) or page_count > 30:
            return 'Renewal_Amendment'
        return 'Amendment'
    return 'Other'


def select_files_for_client(folder_path, files):
    """
    Select which contract files to process for a client.
    Rules:
      1. Classify every file (fast: text extraction only).
      2. Deduplicate same-date files: keep only the largest when any >1 MB.
      3. Split into post-cutoff and pre-cutoff sets.
      4. Always include all post-cutoff files.
      5. If post-cutoff has no MSA/Renewal_Amendment, fall back to most recent
         qualifying pre-cutoff file (2019+).
    """
    def _classify_by_filename(f):
        name_lower = f.lower()
        if any(kw in name_lower for kw in _MSA_NAME_KWS):
            return 'MSA'
        if any(kw in name_lower for kw in _RENEWAL_TEXT_KWS):
            return 'Renewal_Amendment'
        if any(kw in name_lower for kw in _AMEND_NAME_KWS):
            return 'Amendment_maybe'
        return 'Other'

    all_meta = []
    for f in files:
        date    = extract_date_from_filename(f) or datetime(1900, 1, 1)
        size_mb = os.path.getsize(os.path.join(folder_path, f)) / (1024 * 1024)
        all_meta.append({'filename': f, 'date': date, 'size_mb': size_mb,
                         'post': date.year >= DATE_CUTOFF_YEAR,
                         'type_hint': _classify_by_filename(f)})

    post_files = [m for m in all_meta if m['post']]
    pre_files  = [m for m in all_meta if not m['post'] and m['date'].year >= 2019]
    to_classify = post_files if post_files else pre_files
    print(f'  {len(all_meta)} total files — {len(to_classify)} in-range, '
          f'{len(all_meta) - len(to_classify)} skipped (out of date range)')

    has_qualifying_by_name = any(m['type_hint'] in _QUALIFYING_TYPES for m in to_classify)

    meta = []
    for m in to_classify:
        if m['type_hint'] in ('MSA', 'Renewal_Amendment', 'Other'):
            ctype = m['type_hint']
        elif has_qualifying_by_name:
            ctype = 'Amendment'
        else:
            ctype = classify_contract(os.path.join(folder_path, m['filename']))
        meta.append({'filename': m['filename'], 'date': m['date'], 'type': ctype,
                     'post': m['post'], 'size_mb': m['size_mb']})

    date_groups = defaultdict(list)
    for m in meta:
        date_groups[m['date']].append(m)
    deduped = []
    for date, group in date_groups.items():
        if len(group) == 1 or date.year <= 1900:
            deduped.extend(group)
        elif any(m['size_mb'] > 1 for m in group):
            largest  = max(group, key=lambda m: m['size_mb'])
            dropped  = [m['filename'] for m in group if m is not largest]
            date_str = date.strftime('%Y-%m-%d')
            print(f'  [Dedupe] {date_str}: keeping {largest["filename"]} ({largest["size_mb"]:.1f} MB), dropping {dropped}')
            deduped.append(largest)
        else:
            deduped.extend(group)
    meta = deduped

    for m in sorted(meta, key=lambda x: x['date'], reverse=True):
        era      = f'post-{DATE_CUTOFF_YEAR}' if m['post'] else f'pre-{DATE_CUTOFF_YEAR + 1}'
        date_str = m['date'].strftime('%Y-%m-%d') if m['date'].year > 1900 else 'no-date  '
        print(f'    [{era}] {m["type"]:<20s} {date_str}  {m["size_mb"]:5.1f} MB  {m["filename"][:60]}')

    post = [m for m in meta if m['post']]
    pre  = [m for m in meta if not m['post']]
    selected_set = {m['filename'] for m in post}

    if not any(m['type'] in _QUALIFYING_TYPES for m in post):
        if not post:
            reason = f'no contracts dated after {DATE_CUTOFF_YEAR}'
        else:
            reason = f'no MSA/Renewal_Amendment among {len(post)} post-{DATE_CUTOFF_YEAR} file(s)'
        pool = [m for m in pre if m['type'] in _QUALIFYING_TYPES and m['date'].year >= 2019]
        if not pool:
            pool = [m for m in pre if m['type'] == 'Amendment' and m['date'].year >= 2019]
        if pool:
            best = max(pool, key=lambda m: m['date'])
            selected_set.add(best['filename'])
            date_label = best['date'].strftime('%Y-%m-%d') if best['date'].year > 1900 else 'no-date'
            print(f'  [Fallback] {reason} -> adding {best["filename"]} ({best["type"]}, {date_label})')
        else:
            print(f'  [Warning] {reason} and no qualifying pre-cutoff fallback found.')

    selected = [m for m in meta if m['filename'] in selected_set]
    selected.sort(key=lambda m: m['date'], reverse=True)
    print(f'  Selected {len(selected)}/{len(meta)} file(s) for extraction.')
    return [m['filename'] for m in selected]


print('Phase 1 classification helpers loaded.')

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """
You are a forensic-level contract analysis expert.

Your task is to extract ALL items that have an associated price/fee field — INCLUDING items priced at $0, "Included", "Waived", "No Charge", "N/A", "Free", "Complimentary" — AND for each item ALSO capture the Material Code if it is visible on the contract page.

PRICE FIELD — WHAT TO EXTRACT:
You MUST extract every item that has any value in a price/fee column or position, including:
- Dollar amounts: "$100", "$0", "$1,500.00", "USD 250"
- Free / no-cost labels: "Included", "Waived", "No Charge", "N/A", "Free", "Complimentary", "$0"
- TBD / pending: "TBD", "To Be Determined" (capture as the price string, do not skip)
For non-dollar labels (Included, Waived, etc.), set the "price" field to the EXACT label as written on the page.
Items priced at $0 or labeled "Included"/"Waived" still require a material code mapping — do NOT skip them.

CRITICAL INSTRUCTIONS FOR CHECKBOX DETECTION:

1. CHECKBOX PLACEMENT: Checkboxes can appear BEFORE or AFTER the fee description. Look carefully at the LEFT side of each line item for empty or filled boxes.
2. CHECKBOX TYPES:
   - CHECKED: ☑, ☒, ✓, ✗, [X], [x], filled box, box with any mark inside
   - UNCHECKED: ☐, □, [ ], empty box
3. EMPTY BOX before a fee → checkbox_checked: false. MARKED box → true. No checkbox at all → null.
4. HIERARCHICAL: If a parent checkbox is unchecked, all child items are unchecked unless they have their own marked box.

CHECKBOX LOOK-BACK / VISUAL BLOCK / SECTION BOUNDARY RULES:
Same rules as standard Portico extraction — search upward for the closest controlling checkbox in the same visual block; do not cross section boundaries; tables inherit the checkbox above them unless a row has its own.

MATERIAL CODE EXTRACTION:
Material Codes are short alphanumeric identifiers (e.g. GPS022, ACHALERT0001, MS3001) that often appear in contracts directly next to a line item, in a dedicated column, or in parentheses after the description.
- For EACH extracted item, look for a Material Code adjacent to that item (same row, same line, or in a "Material Code" / "Item Code" column).
- If a code is present, capture it EXACTLY as written into the "material_code" field.
- If no code is visible for that item, set material_code to null.
- Material codes are typically 5-12 characters, alphanumeric, often uppercase.

SECTION HEADER IDENTIFICATION AND INHERITANCE

You must determine the SECTION HEADER corresponding to each extracted item.
A Section Header is a high-level title that describes the contract section containing the items. These headers typically appear:
- At the TOP of a page
- CENTERED horizontally — this is the most important visual indicator
- In larger or bold text
- Above paragraphs, lists, or tables
- Often without a dollar value

CRITICAL: Central horizontal alignment is the PRIMARY visual indicator of a Section Header. Text that is left-aligned, indented, or inline with paragraph content is NEVER a Section Header, regardless of its size or formatting.

MULTI-LINE HEADER RULE:
Some section headers span multiple lines or consist of a title and a subtitle directly below it.
If a title line is immediately followed by another descriptive line with no other content between them, treat the COMBINED text as the full section header.
Both lines must be centrally aligned for this rule to apply.

SECTION HEADER ASSOCIATION PROCESS:
STEP 1 — Upward Visual Scan: When extracting an item with a dollar value, scan upward visually from the item's position on the page. Identify the nearest CENTRALLY ALIGNED text block above the item.
STEP 2 — Header Selection: The FIRST qualifying section header encountered while scanning upward is the correct Section Header for that item.
STEP 3 — Page-Level Inheritance: If the current page does NOT contain a section header above the item, inherit the most recent section header that appeared on previous pages.
STEP 4 — Section Scope: A section header governs all paragraphs, bullet lists, numbered clauses, lettered clauses, tables, and fee lines that appear beneath it until a new section header appears.
STEP 5 — Table Handling: If a table appears under a section header, ALL rows in the table inherit the same section header unless another header appears inside the table block.

SECTION HEADER GENERICITY CHECK:
STEP 6 — Genericity Check: A section header is GENERIC (insufficient) if it contains ONLY structural/positional words such as: "Attachment", "Appendix", "Schedule", "Exhibit", "Section", "Fees", "Notes", "Table" — alone or combined with a number or letter, and does NOT contain any specific service name or descriptive topic.
STEP 7 — Predecessor Fallback: If the header is GENERIC, scan upward for the nearest NON-GENERIC centrally aligned header and combine: "[Non-Generic] — [Generic]". If no non-generic predecessor exists, return the generic header as-is.

FAILURE RULES:
- Every extracted item MUST have a Section Header returned. No exceptions.
- Returning null for any item when any header exists anywhere above it in the document is an ERROR.

STRIKETHROUGH HANDLING:
- IGNORE struck values. If a struck price is followed by a new non-struck price, capture only the new one.
- If a non-struck label says "Included" / "Waived" / "$0" / "No Charge" — DO extract the item with that label as the price.

WAIVED-LABEL OVERRIDE RULE:
If the word "waived" (case-insensitive, including "(waived)", "[waived]", or "Fee waived") appears anywhere in or directly adjacent to the price/description for an item, you MUST treat that item as waived.
- Set "price" to "Waived" verbatim (do NOT capture any visible dollar amount alongside it as the price).

PERCENTAGE PRICING:
If an item is priced as a percentage (e.g. "2.9%", "0.05% of transaction amount"), capture the FULL price string as written.

EXCLUSION RULES:
- EXCLUDE totals, subtotals, grand totals.
- EXCLUDE asset-size thresholds, tier triggers (e.g. "over $100M"), prose conditions.

TABLE COLUMN PRIORITY:
- If QTY + UNIT PRICE + AMOUNT all exist: use AMOUNT for price, QTY for quantity, UNITS for frequency. Ignore UNIT PRICE.
- If AMOUNT is TBD/blank: fall back to UNIT PRICE.

TIER ROW SELECTION:
1. ROW-LEVEL CHECKBOXES PRESENT: Only extract rows whose box is MARKED.
2. NO ROW-LEVEL CHECKBOXES: EXTRACT EVERY TIER ROW IN THE TABLE. Do NOT pick only one tier.
3. ABSENCE OF VISIBLE TIER CHECKBOXES IS NOT A SIGNAL TO PICK ONE TIER.

Return STRICT JSON:

{
  "items": [
    {
      "item": "<clear description>",
      "checkbox_checked": true | false | null,
      "price": "<dollar value exactly as written>",
      "quantity": "<qty if present, else null>",
      "frequency": "<frequency / units if present, else null>",
      "page": <page number>,
      "explanation": "brief explanation",
      "section_header": "<section title or null>",
      "material_code": "<material code from contract if visible, else null>"
    }
  ]
}

IMPORTANT:
- Use ONLY visible text. Do NOT infer.
- Be exhaustive. Missing a dollar value is an error.
- Do not include markdown.
"""


def extract_chunk(page_chunk, chunk_index):
    """
    Send one chunk of rendered PDF pages to the proxy and return parsed JSON.
    Uses Chat Completions with base64 image_url (converted from Responses API format).
    ⚠ detail='high' is required for readable contract text; 'low' will miss prices.
    """
    content = []

    ref_img_path = os.path.join(BASE_DIR, 'marked_checkbox_example.png')
    if not os.path.exists(ref_img_path):
        raise FileNotFoundError(
            f'Missing checkbox reference image: {ref_img_path}\n'
            'Copy marked_checkbox_example.png into BASE_DIR before running Phase 1.'
        )
    with open(ref_img_path, 'rb') as f:
        ref_b64 = base64.b64encode(f.read()).decode('utf-8')

    content.append({
        'type': 'text',
        'text': (
            'The first image is a REFERENCE EXAMPLE of a MARKED checkbox. '
            'Any square containing visible crossing diagonal lines is ALWAYS classified as MARKED.'
        ),
    })
    content.append({
        'type': 'image_url',
        'image_url': {'url': f'data:image/png;base64,{ref_b64}', 'detail': 'high'},
    })
    content.append({
        'type': 'text',
        'text': (
            'The following images are sequential pages from a contract. '
            'Use the checkbox reference example above to correctly classify checkboxes.'
        ),
    })

    for page in page_chunk:
        b64 = image_to_base64(page['image'])
        content.append({
            'type': 'image_url',
            'image_url': {'url': f'data:image/png;base64,{b64}', 'detail': 'high'},
        })

    payload = {
        'messages': [
            {'role': 'system', 'content': EXTRACTION_SYSTEM_PROMPT},
            {'role': 'user',   'content': content},
        ],
        'temperature': 0,
    }

    data = call_api_with_retry(lambda: _proxy_post(payload))
    raw  = data['choices'][0]['message']['content'].strip()

    if raw.startswith('```'):
        raw = re.sub(r'^```[a-zA-Z]*\n?', '', raw)
        raw = re.sub(r'```$', '', raw).strip()

    try:
        return json.loads(raw)
    except Exception as e:
        print(f'  ⚠ JSON parse failed in chunk {chunk_index}: {e}')
        print(f'  Preview: {raw[:300]}')
        return {'items': []}


print('Extraction prompt + extract_chunk loaded.')

In [ ]:
def extract_items_with_dollar_values(pdf_path):
    """
    Phase 1: render PDF as images, send chunks to proxy in parallel, collect items.
    No OpenAI client object needed — proxy is stateless.
    """
    print('  Converting PDF to images...')
    pages        = pdf_to_images(pdf_path)
    total_pages  = len(pages)
    total_chunks = math.ceil(total_pages / CHUNK_SIZE)
    n_workers    = min(CHUNK_WORKERS, total_chunks)
    print(f'  Total pages: {total_pages} | Chunks: {total_chunks} | Workers: {n_workers}')

    chunk_results = [None] * total_chunks
    done_count    = [0]
    lock          = threading.Lock()

    def _process_chunk(chunk_index):
        start = chunk_index * CHUNK_SIZE
        chunk = pages[start:start + CHUNK_SIZE]
        for attempt in range(1, 4):
            try:
                result = extract_chunk(chunk, chunk_index + 1)
                with lock:
                    done_count[0] += 1
                    print(f'  Chunk {chunk_index + 1}/{total_chunks} complete '
                          f'({done_count[0]}/{total_chunks} done)')
                return chunk_index, result
            except Exception as e:
                err = str(e).lower()
                transient = any(k in err for k in (
                    'connection', 'timeout', 'temporar', '503', '502', '504', 'reset', 'broken pipe'
                ))
                if attempt < 3 and transient:
                    wait = 10 * attempt
                    print(f'  ⚠ Chunk {chunk_index + 1} attempt {attempt} failed: {e} — retrying in {wait}s')
                    time.sleep(wait)
                else:
                    print(f'  ⚠ Chunk {chunk_index + 1} failed after {attempt} attempt(s): {e} — skipping')
                    with lock:
                        done_count[0] += 1
                    return chunk_index, None
        return chunk_index, None

    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futures = [ex.submit(_process_chunk, i) for i in range(total_chunks)]
        for fut in as_completed(futures):
            ci, parsed = fut.result()
            chunk_results[ci] = parsed

    all_items = []
    for parsed in chunk_results:
        if parsed is None:
            continue
        for item in parsed.get('items', []):
            all_items.append({
                'Item':                    item.get('item'),
                'Price':                   item.get('price'),
                'Quantity':                item.get('quantity'),
                'Frequency':               item.get('frequency'),
                'Page':                    item.get('page'),
                'Checkbox_Checked':        item.get('checkbox_checked'),
                'Explanation':             item.get('explanation'),
                'Section_Header':          item.get('section_header'),
                'Extracted_Material_Code': item.get('material_code'),
            })
    df_items = pd.DataFrame(all_items)
    df_items = _apply_frequency_inference(df_items)
    return df_items


def group_pdfs_by_client(folder_path):
    """
    Detect layout and group PDFs by client.
    - Subdirectories present: each subdir = one client.
    - Flat folder: group by filename prefix (XXX--- pattern).
    Returns dict: { client_name: { 'folder': path, 'files': [filenames] } }
    """
    entries     = os.listdir(folder_path)
    subdirs     = [e for e in entries if os.path.isdir(os.path.join(folder_path, e))]
    direct_pdfs = [e for e in entries if e.lower().endswith('.pdf')]
    groups = {}
    if subdirs:
        for sd in subdirs:
            sub_path = os.path.join(folder_path, sd)
            files    = [f for f in os.listdir(sub_path) if f.lower().endswith('.pdf')]
            if files:
                groups[sd.strip()] = {'folder': sub_path, 'files': files}
    elif direct_pdfs:
        for f in direct_pdfs:
            m = re.match(r'^([A-Z0-9\-]+?)---', f)
            client = m.group(1).replace('-', ' ').strip() if m else os.path.splitext(f)[0]
            grp = groups.setdefault(client, {'folder': folder_path, 'files': []})
            grp['files'].append(f)
    return groups


def process_client(folder_path, client_name, files):
    selected_files = select_files_for_client(folder_path, files)
    if not selected_files:
        print(f'  No files selected for "{client_name}" — skipping.')
        return pd.DataFrame()
    files_with_dates = sorted(
        [(f, extract_date_from_filename(f) or datetime(1900, 1, 1)) for f in selected_files],
        key=lambda x: x[1], reverse=True,
    )
    all_dfs = []
    for file, date_obj in files_with_dates:
        full_path = os.path.join(folder_path, file)
        print(f'\n  Extracting: {file}')
        try:
            df = extract_items_with_dollar_values(full_path)
        except Exception as e:
            print(f'  ⚠ Failed: {e} — skipping')
            continue
        if df.empty:
            print('  No items extracted.')
            continue
        df['Source Contract'] = file
        df['Date'] = date_obj.strftime('%Y-%m-%d') if date_obj.year > 1900 else ''
        all_dfs.append(df)
    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()


def export_to_excel(df, output_path):
    if df.empty:
        print('  No dollar values found — skipping export.')
        return
    with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name='Dollar_Items')
    print(f'  Extraction saved: {output_path}')


print('Phase 1 extraction driver loaded.')

In [ ]:
LEGAL_NOISE = [
    r'pursuant to[^,.]*',
    r'as defined in[^,.]*',
    r'in accordance with[^,.]*',
    r'as set forth[^,.]*',
    r'as described in[^,.]*',
    r'under section [^,.]*',
    r'hereunder',
    r'thereunder',
    r'as applicable',
    r'if applicable',
]


def strip_legal_noise(text):
    if not isinstance(text, str):
        return ''
    s = text
    for p in LEGAL_NOISE:
        s = re.sub(p, '', s, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', s).strip()


def normalize_section(raw_header, norm_lookup, norm_choices):
    if not isinstance(raw_header, str) or not raw_header.strip():
        return None, 0
    key = raw_header.strip().lower()
    if key in norm_lookup:
        return norm_lookup[key], 100
    best = process.extractOne(key, norm_choices, scorer=fuzz.WRatio)
    if best and best[1] >= SECTION_NORM_THRESHOLD:
        return norm_lookup[best[0]], best[1]
    return raw_header.strip(), best[1] if best else 0


def filter_dict_by_section(df_dict, normalized_section):
    if not normalized_section or df_dict.empty:
        return df_dict.iloc[0:0]
    norm_low   = normalized_section.lower()
    sect_lower = df_dict['Section Header'].astype(str).str.lower()
    mask       = sect_lower.str.contains(re.escape(norm_low), na=False)
    sub        = df_dict[mask]
    if not sub.empty:
        return sub
    unique_secs = sect_lower.unique().tolist()
    matches = [s for s in unique_secs if fuzz.partial_ratio(norm_low, s) >= 75]
    if matches:
        return df_dict[sect_lower.isin(matches)]
    return df_dict.iloc[0:0]


MATCHING_SYSTEM_PROMPT = """
You are an expert at matching contract line items to a Material Code dictionary.

For each contract item, return the BEST matching dictionary entry's Contract Description (verbatim) and its Material Code.

RULES:
1. Match on MEANING, not surface form. "Mthly" == "Monthly". "Setup" == "Implementation".
2. Use ONLY descriptions that appear in the dictionary block below. Do NOT invent.
3. Monthly fees and one-time/setup fees are not interchangeable.
4. Product specificity beats generic. Prefer the specific product/service over a generic category.
5. If a contract item clearly matches none of the descriptions, return null.

Return STRICT JSON:

{
  "matches": [
    {
      "item_index": <integer index from the input list>,
      "matched_description": "<exact dictionary description or null>",
      "material_code": "<material code from dictionary or null>",
      "confidence": <integer 0-100>
    }
  ]
}

No markdown. No explanations.
"""


def ai_match_batch(item_batch, dict_chunk):
    """Send a batch of items + a chunk of the dictionary to the proxy and parse matches."""
    items_text = '\n'.join(
        f'[{i}] Item: {it["Item"]} | Section: {it.get("Normalized_Section") or it.get("Section_Header") or "N/A"}'
        for i, it in enumerate(item_batch)
    )
    dict_text = '\n'.join(
        f'- [{r["Material Code"]}] {r["Contract Description"]} (Section: {r["Section Header"]})'
        for _, r in dict_chunk.iterrows()
    )
    user_prompt = (
        f'DICTIONARY ENTRIES:\n{dict_text}\n\n'
        f'CONTRACT ITEMS TO MATCH:\n{items_text}\n\n'
        'Return matches for each item by item_index. Use null when no good match exists.'
    )
    payload = {
        'messages': [
            {'role': 'system', 'content': MATCHING_SYSTEM_PROMPT},
            {'role': 'user',   'content': user_prompt},
        ],
        'temperature': 0,
    }
    data = call_api_with_retry(lambda: _proxy_post(payload))
    raw  = data['choices'][0]['message']['content'].strip()
    if raw.startswith('```'):
        raw = re.sub(r'^```[a-zA-Z]*\n?', '', raw)
        raw = re.sub(r'```$', '', raw).strip()
    try:
        return json.loads(raw).get('matches', [])
    except Exception as e:
        print(f'  ⚠ AI match JSON parse failed: {e}')
        return []


def hybrid_score_item(item_text, candidate_idxs, item_emb, all_desc_emb, all_desc_texts):
    if not candidate_idxs:
        return None, 0.0
    cand_emb   = all_desc_emb[candidate_idxs]
    sem_scores = (item_emb @ cand_emb.T).flatten()
    cand_texts = [all_desc_texts[i] for i in candidate_idxs]
    try:
        vec   = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
        tfidf = vec.fit_transform([item_text] + cand_texts)
        lex   = (tfidf[0] @ tfidf[1:].T).toarray().flatten()
        if lex.max() > 0:
            lex = lex / lex.max()
    except Exception:
        lex = np.zeros_like(sem_scores)
    combined   = SEMANTIC_WEIGHT * sem_scores + LEXICAL_WEIGHT * lex
    best_local = int(np.argmax(combined))
    return candidate_idxs[best_local], float(combined[best_local])


def run_matching(extraction_df, df_dict, df_norm):
    """Phase 2: section normalization + section-aware matching with precomputed embeddings."""
    print('Loading semantic model...')
    semantic_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
    print('Semantic model loaded.')

    df_norm_clean = df_norm[['Section Headers', 'Section']].dropna()
    norm_lookup   = {
        str(r['Section Headers']).strip().lower(): str(r['Section']).strip()
        for _, r in df_norm_clean.iterrows()
    }
    norm_choices = list(norm_lookup.keys())
    print(f'Normalization table: {len(norm_lookup)} mappings')
    print(f'Dictionary: {len(df_dict)} rows, {df_dict["Section Header"].nunique()} unique sections')

    df = extraction_df.copy()
    df = df[df['Checkbox_Checked'].isin([True, None]) | df['Checkbox_Checked'].isna()]
    df = df[df['Item'].notna() & (df['Item'].astype(str).str.strip() != '')]
    df = df.reset_index(drop=True)

    print(f'Normalizing section headers for {len(df)} items...')
    norm_results         = df['Section_Header'].apply(
        lambda h: normalize_section(h, norm_lookup, norm_choices)
    )
    df['Normalized_Section'] = [r[0] for r in norm_results]
    df['Section_Norm_Score'] = [r[1] for r in norm_results]

    print(f'Encoding {len(df_dict)} dictionary descriptions...')
    df_dict    = df_dict.reset_index(drop=True)
    dict_descs = df_dict['Contract Description'].astype(str).tolist()
    dict_emb   = semantic_model.encode(dict_descs, convert_to_numpy=True,
                                       normalize_embeddings=True, batch_size=64,
                                       show_progress_bar=False)

    print(f'Encoding {len(df)} contract items...')
    item_texts = [strip_legal_noise(t or '') for t in df['Item'].astype(str).tolist()]
    item_embs  = semantic_model.encode(item_texts, convert_to_numpy=True,
                                       normalize_embeddings=True, batch_size=64,
                                       show_progress_bar=False)

    print('Building section -> dict-row index map...')
    sect_lower_series = df_dict['Section Header'].astype(str).str.lower()
    section_index_cache = {}

    def get_section_indices(normalized_section):
        if not normalized_section or not isinstance(normalized_section, str):
            return None
        key = normalized_section.lower()
        if key in section_index_cache:
            return section_index_cache[key]
        mask = sect_lower_series.str.contains(re.escape(key), na=False)
        idxs = df_dict.index[mask].tolist()
        if not idxs:
            unique_secs  = sect_lower_series.unique().tolist()
            matched_secs = [s for s in unique_secs if fuzz.partial_ratio(key, s) >= 75]
            if matched_secs:
                idxs = df_dict.index[sect_lower_series.isin(matched_secs)].tolist()
        section_index_cache[key] = idxs
        return idxs

    print('Building product-keyword anchor index...')
    raw_secs        = df_dict['Section Header'].dropna().astype(str).unique().tolist()
    anchor_keywords = set()
    for s in raw_secs:
        s_clean = s.strip()
        if re.fullmatch(r'[A-Z][A-Z0-9 \-]{2,20}', s_clean) and len(s_clean.split()) <= 3:
            for w in s_clean.split():
                if len(w) >= 4 and w.isalpha():
                    anchor_keywords.add(w.lower())
        m = re.match(r'^([A-Z][a-z]{4,15})(\s|$|\u00ae|\u2122)', s_clean)
        if m:
            anchor_keywords.add(m.group(1).lower())
    anchor_keywords.update({
        'wisdom', 'mobiliti', 'zelle', 'alldata', 'nautilus', 'cardhub',
        'originate', 'notifi', 'weiland', 'checkfree', 'convergeit',
        'loancierge', 'transfernow', 'fundnow', 'securenow', 'premier',
        'signature', 'directors', 'data compass', 'data safe',
        'card valet', 'instant open', 'tmagic', 'openchecking',
        'card expert', 'card risk office', 'configure digital',
        'kinective', 'intelligent workplace',
    })
    print(f'  Anchor keywords: {len(anchor_keywords)}')

    keyword_index_cache = {}
    sect_low = sect_lower_series.values
    desc_low = df_dict['Contract Description'].astype(str).str.lower().values
    for kw in anchor_keywords:
        mask = (
            pd.Series(sect_low).str.contains(re.escape(kw), na=False)
            | pd.Series(desc_low).str.contains(re.escape(kw), na=False)
        )
        keyword_index_cache[kw] = df_dict.index[mask].tolist()

    def get_keyword_indices(item_text):
        if not item_text:
            return []
        t    = item_text.lower()
        idxs = set()
        for kw, kw_idxs in keyword_index_cache.items():
            if kw in t:
                idxs.update(kw_idxs)
        return list(idxs)

    print(f'Scoring {len(df)} items via hybrid (sentence-transformers + TF-IDF)...')
    results    = [None] * len(df)
    ai_pending = []
    full_idxs  = df_dict.index.tolist()

    for i, row in df.iterrows():
        section        = row.get('Normalized_Section')
        extracted_code = row.get('Extracted_Material_Code')

        if isinstance(extracted_code, str) and extracted_code.strip():
            code_clean = extracted_code.strip().upper()
            cand_idxs  = get_section_indices(section) or full_idxs
            sub        = df_dict.loc[cand_idxs]
            hit        = sub[sub['Material Code'].astype(str).str.upper() == code_clean]
            label      = 'Contract+Dict (section)'
            if hit.empty:
                hit   = df_dict[df_dict['Material Code'].astype(str).str.upper() == code_clean]
                label = 'Contract+Dict (full)'
            if not hit.empty:
                r = hit.iloc[0]
                results[i] = {
                    'Final Material Code': r['Material Code'],
                    'Matched Description': r['Contract Description'],
                    'Match Source':        label,
                    'Confidence Percentage': 100,
                }
            else:
                results[i] = {
                    'Final Material Code': extracted_code,
                    'Matched Description': '',
                    'Match Source':        'Contract-only (unvalidated)',
                    'Confidence Percentage': 90,
                }
            continue

        section_idxs = get_section_indices(section) or []
        keyword_idxs = get_keyword_indices(item_texts[i])
        pool_idxs    = list(set(section_idxs) | set(keyword_idxs))

        if pool_idxs:
            using_full = False
        else:
            using_full = True
            pool_idxs  = full_idxs

        if not item_texts[i]:
            results[i] = {'Final Material Code': '', 'Matched Description': '',
                          'Match Source': 'Empty item', 'Confidence Percentage': 0}
            continue

        best_idx, score = hybrid_score_item(
            item_texts[i], pool_idxs, item_embs[i], dict_emb, dict_descs
        )

        if score >= FUZZY_AUTO_ACCEPT_THRESHOLD and best_idx is not None:
            r = df_dict.iloc[best_idx]
            results[i] = {
                'Final Material Code': r['Material Code'],
                'Matched Description': r['Contract Description'],
                'Match Source':        'Hybrid (auto)' + (' - full' if using_full else ''),
                'Confidence Percentage': int(round(score * 100)),
            }
        else:
            ai_pending.append((i, pool_idxs, using_full, score, best_idx))

    print(f'  Hybrid auto-accepted: {sum(1 for r in results if r and r["Match Source"].startswith("Hybrid"))}')
    print(f'  Contract-validated:   {sum(1 for r in results if r and r["Match Source"].startswith("Contract"))}')
    print(f'  Pending AI:           {len(ai_pending)}')

    if ai_pending:
        # No OpenAI client object needed — proxy is stateless
        def run_ai_against(cand_idxs_, row_idx_):
            cand_df = df_dict.loc[cand_idxs_].head(DICT_CHUNK_SIZE)
            try:
                ai_results = ai_match_batch([df.iloc[row_idx_].to_dict()], cand_df)
                if ai_results:
                    m = ai_results[0]
                    return {
                        'material_code':       m.get('material_code') or '',
                        'matched_description': m.get('matched_description') or '',
                        'confidence':          m.get('confidence') or 0,
                    }
            except Exception as e:
                print(f'  ⚠ AI match failed for row {row_idx_}: {e}')
            return None

        def ai_one(row_idx, cand_idxs, using_full, fallback_score, fallback_idx):
            primary = run_ai_against(cand_idxs, row_idx)
            best  = primary
            label = 'AI' + (' - full' if using_full else '')
            if not using_full and (not primary or primary.get('confidence', 0) < 70):
                full_top_idxs = list(np.argsort(
                    SEMANTIC_WEIGHT * (item_embs[row_idx] @ dict_emb.T)
                )[::-1][:DICT_CHUNK_SIZE])
                full_ai = run_ai_against(full_top_idxs, row_idx)
                if full_ai and full_ai.get('confidence', 0) > (primary.get('confidence', 0) if primary else 0):
                    best  = full_ai
                    label = 'AI - full (fallback)'
            if best:
                return row_idx, {
                    'Final Material Code': best.get('material_code', ''),
                    'Matched Description': best.get('matched_description', ''),
                    'Match Source':        label,
                    'Confidence Percentage': best.get('confidence', 0),
                }
            if fallback_idx is not None:
                r = df_dict.iloc[fallback_idx]
                return row_idx, {
                    'Final Material Code': r['Material Code'],
                    'Matched Description': r['Contract Description'],
                    'Match Source':        'Hybrid (fallback)' + (' - full' if using_full else ''),
                    'Confidence Percentage': int(round(fallback_score * 100)),
                }
            return row_idx, {'Final Material Code': '', 'Matched Description': '',
                             'Match Source': 'No match', 'Confidence Percentage': 0}

        with ThreadPoolExecutor(max_workers=MAX_PARALLEL_CALLS) as ex:
            futures = [ex.submit(ai_one, *p) for p in ai_pending]
            done    = 0
            for fut in as_completed(futures):
                row_idx, res = fut.result()
                results[row_idx] = res
                done += 1
                if done % 25 == 0:
                    print(f'  AI matched {done}/{len(ai_pending)}')

    df_match = pd.DataFrame(results)
    df_out   = pd.concat([df.reset_index(drop=True), df_match.reset_index(drop=True)], axis=1)
    df_out['Cleaned Price'] = df_out['Price'].apply(clean_price)
    df_out['Pricing_Type']  = df_out['Price'].apply(detect_pricing_type)
    df_out['Tier_Range']    = df_out['Item'].apply(extract_tier_range)
    if 'Section_Norm_Score' in df_out.columns:
        df_out = df_out.drop(columns=['Section_Norm_Score'])
    return df_out


print('Phase 2 matching functions loaded.')

In [ ]:
# Preflight: check checkbox reference image exists before wasting time on extraction
_ref_png = os.path.join(BASE_DIR, 'marked_checkbox_example.png')
if not os.path.exists(_ref_png):
    raise FileNotFoundError(
        f'Missing checkbox reference image: {_ref_png}\n'
        'Copy marked_checkbox_example.png into BASE_DIR before running Phase 1.'
    )
print(f'Checkbox reference image: OK ({_ref_png})')

print('=' * 60)
print('DNA Pipeline v4 — Fiserv Foundation API (keyless)')
print(f'  Date filter: contracts after {DATE_CUTOFF_YEAR}')
print(f'  DPI: {DPI}  |  Chunk workers: {CHUNK_WORKERS}')
print(f'  Email ID: {EMAIL_ID}')
print('=' * 60)

groups = group_pdfs_by_client(CONTRACTS_DIR)
print(f'Found {len(groups)} client group(s):')
for c, info in groups.items():
    print(f'  - {c}: {len(info["files"])} file(s) in {info["folder"]}')

if ONLY_CLIENT:
    groups = {k: v for k, v in groups.items() if k == ONLY_CLIENT}
    if not groups:
        raise SystemExit(f'Client "{ONLY_CLIENT}" not found in {CONTRACTS_DIR}')

print('\nLoading DNA dictionary + normalization tables...')
df_dict = pd.read_excel(DICTIONARY_FILE, sheet_name='Sheet1')
df_dict = df_dict.dropna(subset=['Material Code', 'Contract Description'])
df_norm = pd.read_excel(NORM_FILE, sheet_name='Sheet2')
print(f'Dictionary: {len(df_dict)} rows loaded.')
print(f'Norm table: {len(df_norm)} rows loaded.')

In [ ]:
# Phase 1 — Extract dollar items from selected contracts per client.
# Skips clients where extraction output already exists.
# Delete the output file to force a re-run for a specific client.

for client_name, info in groups.items():
    client_folder = info['folder']
    files         = info['files']
    print(f"\n{'=' * 60}\nClient: {client_name}\n{'=' * 60}")

    extracted_path = os.path.join(OUTPUT_DIR, f'Tool Extraction {client_name} Final.xlsx')

    if os.path.exists(extracted_path):
        print(f'  Phase 1 — using existing extraction: {os.path.basename(extracted_path)}')
    else:
        df_extract = process_client(client_folder, client_name, files)
        export_to_excel(df_extract, extracted_path)

print('\nPhase 1 complete.')

In [ ]:
# Phase 2 — Match extracted items to DNA dictionary.
# Skips clients where matched output already exists.
# Run Cell 11 first (or ensure extraction files exist in OUTPUT_DIR).

for client_name, info in groups.items():
    print(f"\n{'=' * 60}\nClient: {client_name}\n{'=' * 60}")

    extracted_path = os.path.join(OUTPUT_DIR,         f'Tool Extraction {client_name} Final.xlsx')
    matched_path   = os.path.join(MATCHED_OUTPUT_DIR, f'Matched_{client_name}.xlsx')

    if not os.path.exists(extracted_path):
        print('  No extraction file found — run Phase 1 first.')
        continue

    df_extract = pd.read_excel(extracted_path, sheet_name='Dollar_Items')
    if df_extract.empty:
        print('  No items — skipping.')
        continue

    if os.path.exists(matched_path):
        print(f'  Phase 2 — output already exists, skipping: {os.path.basename(matched_path)}')
        continue

    df_matched = run_matching(df_extract, df_dict, df_norm)
    with pd.ExcelWriter(matched_path, engine='xlsxwriter') as w:
        df_matched.to_excel(w, index=False, sheet_name='Matched')
    print(f'  Matched output saved: {matched_path}')

print('\nPhase 2 complete.')